# Digital Payment Transaction Analytics & Payment Failure Analysis

Synthetic portfolio/training data only—not actual Paymentus data. Success/failure rate denominator is Success + Failed; Pending excluded.

In [ ]:
import pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\ndf = pd.read_csv('payments.csv')\ndf.head()

## Data exploration

In [ ]:
df.shape\ndf.info()\ndf.describe()\ndf.isnull().sum()\ndf.duplicated().sum()\ndf['transaction_id'].duplicated().sum()

## Data validation

In [ ]:
df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors='coerce')\ndf[df['amount'] <= 0].head()

## Status distribution

In [ ]:
status_counts = df['transaction_status'].value_counts()\nstatus_counts

In [ ]:
status_counts.plot(kind='bar', figsize=(8,5))\nplt.title('Transaction Status Distribution')\nplt.xlabel('Status'); plt.ylabel('Count'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## Overall rates (Pending excluded)

In [ ]:
completed = df[df['transaction_status'].isin(['Success','Failed'])]\nsuccesses = (completed['transaction_status']=='Success').sum()\nfailures = (completed['transaction_status']=='Failed').sum()\neligible = len(completed)\nprint('Success rate:', round(successes/eligible*100,2))\nprint('Failure rate:', round(failures/eligible*100,2))

## Method analysis

In [ ]:
pm = df.groupby('payment_method').agg(total=('transaction_id','count'),successes=('transaction_status',lambda s:(s=='Success').sum()),failures=('transaction_status',lambda s:(s=='Failed').sum())).reset_index()\npm['eligible']=pm['successes']+pm['failures']\npm['success_rate']=pm['successes']/pm['eligible']*100\npm.sort_values('success_rate',ascending=False)

In [ ]:
pm.set_index('payment_method')['success_rate'].plot(kind='bar',figsize=(8,5))\nplt.title('Success Rate by Payment Method'); plt.ylabel('Success Rate (%)'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## Merchant analysis

In [ ]:
merchant = df.groupby('merchant_id').agg(records=('transaction_id','count'),successes=('transaction_status',lambda s:(s=='Success').sum()),failures=('transaction_status',lambda s:(s=='Failed').sum()),amount=('amount','sum')).reset_index()\nmerchant['eligible']=merchant['successes']+merchant['failures']\nmerchant['failure_rate']=merchant['failures']/merchant['eligible']*100\nfailed_value=df[df['transaction_status']=='Failed'].groupby('merchant_id')['amount'].sum()\nmerchant['failed_value']=merchant['merchant_id'].map(failed_value).fillna(0)\nmerchant.sort_values('failure_rate',ascending=False).head(10)

## Failure reasons

In [ ]:
failure=df[df['transaction_status']=='Failed'].groupby('failure_reason').agg(count=('transaction_id','count'),value=('amount','sum')).reset_index().sort_values('count',ascending=False)\nfailure

In [ ]:
failure.set_index('failure_reason')['count'].plot(kind='bar',figsize=(9,5))\nplt.title('Failed Transactions by Failure Reason'); plt.ylabel('Count'); plt.xticks(rotation=45,ha='right'); plt.tight_layout(); plt.show()

## Daily trends

In [ ]:
df['day']=df['transaction_date'].dt.date\ndaily=df.groupby('day').agg(total=('transaction_id','count'),successes=('transaction_status',lambda s:(s=='Success').sum()),failures=('transaction_status',lambda s:(s=='Failed').sum())).reset_index()\ndaily['eligible']=daily['successes']+daily['failures']\ndaily['success_rate']=daily['successes']/daily['eligible']*100\ndaily.head()

In [ ]:
daily.plot(x='day',y='total',kind='line',figsize=(12,5))\nplt.title('Daily Transaction Volume'); plt.ylabel('Transactions'); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

## Failed value by channel

In [ ]:
channel=df[df['transaction_status']=='Failed'].groupby('payment_channel').agg(count=('transaction_id','count'),value=('amount','sum')).reset_index().sort_values('value',ascending=False)\nchannel

In [ ]:
channel.set_index('payment_channel')['value'].plot(kind='bar',figsize=(8,5))\nplt.title('Failed Transaction Value by Channel'); plt.ylabel('Value (CAD)'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## Top merchants by successful transaction value

In [ ]:
df[df['transaction_status']=='Success'].groupby('merchant_id')['amount'].sum().sort_values(ascending=False).head(5)

## Interpretation & limitations

Answer: method with best success rate; dominant failure reason; high-failure merchants and volume context; daily trend; channel with largest failed value; two recommendations. Use actual notebook outputs. Failed transaction value is not necessarily lost revenue; payments may be retried or recovered. Synthetic data, record-level metrics and Pending excluded.